In [132]:
from __future__ import annotations

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, date
from pathlib import Path
import json
import logging

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100


### logging
biar print auto format


In [ ]:
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
logger = logging.getLogger('log_fx')

testing logging

In [7]:
logger.info('shutdown dalam 5 menit')

[INFO] shutdown dalam 5 menit


In [ ]:
class ForexDataError(Exception):
# base
    pass

class InvalidCandleError(ForexDataError):
    pass

class RepositoryNotLoadedError(ForexDataError): # repo diakses sebelum data di load
    pass

class AnalysisError(ForexDataError):
    pass



In [108]:
@dataclass(frozen=True)
class Candle:
    date: datetime
    open: float
    high: float
    low: float
    close: float
    volume: float

    def __post_init__(self) -> None:
        if not isinstance(self.date , datetime):
            raise InvalidCandleError(
                f'tipe data harus date, dapat {type(self.date).__name__}'
            )
    
        for nm in ('open','high','low','close'):
            v = getattr(self , nm) #error handling
            if v <= 0:
                raise InvalidCandleError(f'{nm} harus > 0, v - {v}')
            if self.high < self.low:
                raise InvalidCandleError(f' high {self.high} harus > low {self.low}')
        if not (self.low <= self.open <= self.high):
            raise InvalidCandleError(
                f'open {self.open} harus berada di dalam high dan low'
            )
        if not (self.low <= self.close <= self.high):
            raise InvalidCandleError(
                f'close {self.close} harus berada di dalam high dan low'
            )
        if self.volume < 0:
            raise InvalidCandleError(f'volume harus >= 0')
        
    #helper
    def body_size(self) -> float:
        return abs(self.close - self.open)
    
    def range_size(self) -> float:
        return self.high - self.low
    
    def is_bullish(self) -> bool:
        return self.close > self.open #kalau true = bullish
    
    def daily_return_pct(self) -> float: #return harian
        return (self.close - self.open) / self.open * 100
    
    # magic method
    def __str__(self) -> str:
        tag = 'bull' if self.is_bullish() else 'bear'
        return (
            f'candle({self.date.strftime('%Y-%m-%d')} [{tag}])'
            f'O= {self.open:.5f} H={self.high:.5f}'
            f'L={self.low:.5f} C={self.close:.5f}'
        )
    def __lt__(self, other: "Candle") -> bool: #bandingin candle berdasarkan tanggal
        if not isinstance(other,Candle):
            return NotImplemented
        return self.date < other.date




### testing

In [74]:
assert issubclass(InvalidCandleError, ForexDataError)
assert issubclass(RepositoryNotLoadedError, ForexDataError)
assert issubclass(AnalysisError, ForexDataError)
assert issubclass(ForexDataError, ForexDataError)

## Candle

### bullish candle

In [75]:
c_bull = Candle(date=datetime(2021,1,4),open=1.22340,high=1.23115,low=1.22240,close=1.2290,volume=125000)
assert c_bull.is_bullish() is True, "ini bearish"
assert c_bull.body_size() == (1.2290 - 1.22340)
assert c_bull.range_size() == (1.23115 - 1.22240)
assert c_bull.daily_return_pct() > 0

### bearish candle

In [127]:
c_bear = Candle(date=datetime(2021,1,5), open= 1.23000, high = 1.23100, low = 1.22000, close = 1.22500, volume = 98000)
assert c_bear.is_bullish() == False, "ini bullish"
assert c_bear.daily_return_pct() < 0
print((c_bear.close - c_bear.open) / c_bear.open * 100)



-0.4065040650406418


## testing sorting

In [80]:
candles = [c_bear, c_bull]
sortedCandle = sorted(candles)
assert sortedCandle[0] is c_bull
assert sortedCandle[1] is c_bear

### testing harga negatif

In [ ]:
try:
    Candle(date=datetime(2021,1,5), open= -1, high = 1.5, low = 0.5, close = 1., volume = 100)
    raise AssertionError('fail')
except InvalidCandleError:
    pass

### testing high < low

In [103]:
try:
    Candle(date=datetime(2021,1,7), open = 2, high = 2.5, low = 1.5, close = 1, volume = 100)
    raise AssertionError('valid : lempar ke invalidcandleerror')
except InvalidCandleError as e:
    pass
    print(f'output {e}')

output open 1 harus berada di dalam high dan low


In [111]:
try:
    Candle(date=datetime(2021, 1, 8), open=1.0, high=1.5, low=0.8, close=2.0, volume=100.0)
    raise AssertionError("Seharusnya raise InvalidCandleError")
except InvalidCandleError as e:
    pass
    print(f'pesan error {e}')

pesan error close 2.0 harus berada di dalam high dan low


### testing immutability

In [ ]:
try:
    c_bull.open = 99.0  
    raise AssertionError("dataclass harusnya gabisa dimutate")
except Exception as e:
    print(f'tes {e}')   

tes cannot assign to field 'open'


### tes magic method

In [125]:
s_bull = str(c_bull)
s_bear = str(c_bear)
assert "2021-01-04" in s_bull and "bull" in s_bull, 'error'
assert "2021-01-05" in s_bear and "bear" in s_bear

In [129]:
logger.info('testing output')

[INFO] testing output


### CandleRepository
Menerapkan Repository pattern, rincian logic:
1. Load CSV dengan `pandas.read_csv`
2. Parse Tanggal menjadi `datetime`
3. sort by tanggal

In [ ]:
class CandleRepository:
    _DATE_FORMAT = '%Y.%m.%d %H:%M'
    _CSV_SEPERATOR = ";"

    def __init__(self) -> None:
        self.candles: list[Candle] = []
        self._file_path: str | None = None
    # load data
    def load_csv(self, file_path: str) -> int:
        path = Path(file_path)
        if not path.exists():
            raise FileNotFoundError(f'file tidak ditemukan : {file_path}')
            
        df = pd.read_csv(path, self._CSV_SEPERATOR)
        logger.info(f"berhasil load {len(df):} baris dari {path.name}")

TypeError: CandleRepository.load_csv() missing 1 required positional argument: 'file_path'